<a href="https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/Copy_of_w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/Khadija-Azam05/ML-Projects.git

Cloning into 'ML-Projects'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 172 (delta 69), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 1.88 MiB | 17.34 MiB/s, done.
Resolving deltas: 100% (69/69), done.


In [ ]:
import os

print(os.path.exists("ML-Projects/data/raw/content_refresh_anonymized.csv"))

True


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import pandas as pd

df = pd.read_csv("ML-Projects/data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)

key_fields = [
    "impressions_90d",
    "clicks_90d",
    "days_since_last_update",
    "avg_position",
    "trend_pct"
]

print("\nKey field distributions:")
print(df[key_fields].describe().T)

Shape: (30000, 44)

Key field distributions:
                          count         mean           std    min   25%  \
impressions_90d         30000.0  5200.366300  16838.019547    1.0  81.0   
clicks_90d              30000.0    16.097333     75.076958    0.0   0.0   
days_since_last_update  30000.0    46.098300     42.078709    1.0  20.0   
avg_position            30000.0    16.342380     15.216790    0.0   6.2   
trend_pct               26612.0    -4.785969    473.861780 -100.0 -62.6   

                          50%      75%       max  
impressions_90d         731.0  3615.25  517715.0  
clicks_90d                1.0     7.00    4178.0  
days_since_last_update   20.0   104.00     373.0  
avg_position             10.8    22.30     245.0  
trend_pct               -33.5     0.00   44900.0  


### 1. Distributions

The key fields are strongly right-skewed and contain heavy tails. For example, **impressions_90d** has a median of 731 but a maximum of 517,715, while **clicks_90d** has a median of 1 and a maximum of 4,178. **trend_pct** is also highly variable, with a median of -33.5% and a maximum of 44,900%. These distributions show that a small number of pages have much larger values than most pages, so raw values should be interpreted carefully when building a ranking or decision-support rule.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
stale_bucket = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 60, 90, 180, float("inf")],
    labels=["0-30", "31-60", "61-90", "91-180", "181+"]
)

stale_test = (
    df.assign(stale_bucket=stale_bucket)
      .groupby("stale_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean")
      )
      .reset_index()
)

print(stale_test)

  stale_bucket      n  avg_impressions  avg_clicks
0         0-30  20480      4199.614062   13.727393
1        31-60    128      8323.695312   12.742188
2        61-90     47      1558.468085    1.361702
3       91-180   9171      7486.665140   21.766765
4         181+    174      1172.448276    2.672414


### Signal #1: Days since last update — Mixed

I checked whether pages that have not been updated for longer tend to have higher or lower search activity. The results were mixed: pages updated 91–180 days ago had the highest average impressions and clicks, while pages older than 180 days had much lower activity. Because the pattern does not consistently increase or decrease with update age, I would not use staleness alone to decide which pages need a refresh.


In [ ]:
impression_bucket = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 1000, 5000, float("inf")],
    labels=["0-100", "101-500", "501-1000", "1001-5000", "5000+"]
)

impression_test = (
    df.assign(impression_bucket=impression_bucket)
      .groupby("impression_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_clicks=("clicks_90d", "mean"),
          avg_position=("avg_position", "mean")
      )
      .reset_index()
)

print(impression_test)

  impression_bucket     n  avg_clicks  avg_position
0             0-100  8006    0.146765     13.685336
1           101-500  5279    0.586475     22.161356
2          501-1000  3206    1.421397     20.033624
3         1001-5000  7359    6.306563     15.983503
4             5000+  6150   69.541789     13.311610


### Signal #2: 90-day impressions — Confirmed

I checked whether pages with more impressions also tend to receive more clicks. The pattern is very clear: average clicks increased from 0.15 for pages with 0–100 impressions to 69.54 for pages with more than 5,000 impressions. Based on this data, impressions are a useful directional signal for identifying pages with meaningful search visibility.


In [ ]:
position_bucket = pd.cut(
    df["avg_position"],
    bins=[-1, 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

position_test = (
    df.assign(position_bucket=position_bucket)
      .groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_clicks=("clicks_90d", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

print(position_test)

  position_bucket      n  avg_clicks  avg_impressions
0             1-3   2346   15.792413      3223.757033
1            4-10  11842   26.340821      7546.142543
2           11-20   7273   10.937990      3137.629589
3           21-50   7225    7.457855      4849.645952
4             50+   1314    0.386606       934.522831


### Signal #3: Average position — CONFIRMED

I checked whether search position is related to clicks. The pattern is mostly clear: pages in positions 4–10 had the highest average clicks at 26.34, while pages in positions 50+ had only 0.39 average clicks. This suggests that search position is a useful directional signal when looking for content opportunities, although it should not be used on its own.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# Flag-linked test: does older content show weaker activity?

flag_age_bucket = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, float("inf")],
    labels=["0-90", "91-180", "181+"]
)

flag_test = (
    df.assign(flag_age_bucket=flag_age_bucket)
      .groupby("flag_age_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean"),
          avg_position=("avg_position", "mean")
      )
      .reset_index()
)

print(flag_test)

  flag_age_bucket      n  avg_impressions  avg_clicks  avg_position
0            0-90  20655      4219.161317   13.693149     15.692394
1          91-180   9171      7486.665140   21.766765     17.901461
2            181+    174      1172.448276    2.672414     11.325862


### 3. Flag-linked test — Mixed

I tested the assumption behind a staleness-based refresh flag by comparing pages based on how long it had been since their last update. Pages updated 91–180 days ago actually had the highest average impressions and clicks, while pages older than 180 days had much lower activity. This suggests that update age alone does not show whether a page needs a refresh, so staleness is better treated as a supporting signal rather than a decision by itself.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### 4. What this means in practice

The data suggests that content teams should prioritize pages using a combination of signals rather than relying on staleness alone. Impressions and search position are useful directional signals because they are clearly related to clicks, while update age gives a more mixed picture. A practical refresh queue should therefore use visibility and position to identify opportunities, with staleness used as additional context.
